# Trabajo Práctico Final — Clasificación de Emociones
## Paso 4: Evaluación y Comparación de Modelos

**Métricas implementadas:**
- Curvas ROC y AUC por clase (estrategia One-vs-Rest)
- Matrices de confusión normalizadas y absolutas
- Radar chart de F1-score por emoción
- Tabla comparativa de todos los modelos

---

## 4.0 — Imports y carga de predicciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    classification_report, accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize

BASE_DIR = Path(r'C:\Users\Uriel\Documents\UTN\PIYS')
CLASSES  = ['angry','disgusted','fearful','happy','neutral','sad','surprised']
N_CLS    = len(CLASSES)

EMOTION_COLORS = {
    'angry':'#E74C3C','disgusted':'#8E44AD','fearful':'#2980B9',
    'happy':'#F1C40F','neutral':'#95A5A6','sad':'#2C3E50','surprised':'#E67E22'
}
PALETTE = list(EMOTION_COLORS.values())

# Carga de predicciones de todos los modelos
models_data = {
    'CNN (MiniEmotionNet)': {
        'y_true': np.load(BASE_DIR / 'cnn_y_true.npy'),
        'y_pred': np.load(BASE_DIR / 'cnn_y_pred.npy'),
        'y_prob': np.load(BASE_DIR / 'cnn_y_prob.npy'),
        'color': 'steelblue',
    },
    'ResNet18 (Transfer Learning)': {
        'y_true': np.load(BASE_DIR / 'resnet_y_true.npy'),
        'y_pred': np.load(BASE_DIR / 'resnet_y_pred.npy'),
        'y_prob': np.load(BASE_DIR / 'resnet_y_prob.npy'),
        'color': 'darkorange',
    },
    'XGBoost + CNN features': {
        'y_true': np.load(BASE_DIR / 'xgb_cnn_y_true.npy'),
        'y_pred': np.load(BASE_DIR / 'xgb_cnn_y_pred.npy'),
        'y_prob': np.load(BASE_DIR / 'xgb_cnn_y_prob.npy'),
        'color': 'seagreen',
    },
    'XGBoost + ResNet features': {
        'y_true': np.load(BASE_DIR / 'xgb_rn_y_true.npy'),
        'y_pred': np.load(BASE_DIR / 'xgb_rn_y_pred.npy'),
        'y_prob': np.load(BASE_DIR / 'xgb_rn_y_prob.npy'),
        'color': 'purple',
    },
    'XGBoost + RFECV': {
        'y_true': np.load(BASE_DIR / 'xgb_sel_y_true.npy'),
        'y_pred': np.load(BASE_DIR / 'xgb_sel_y_pred.npy'),
        'y_prob': np.load(BASE_DIR / 'xgb_sel_y_prob.npy'),
        'color': 'crimson',
    },
}

print('Modelos cargados:')
for name, d in models_data.items():
    acc = accuracy_score(d['y_true'], d['y_pred'])
    print(f'  {name:40s} Acc: {acc:.4f}')

## 4.1 — Curvas ROC y AUC por clase (One-vs-Rest)

Para clasificación multiclase, computamos la curva ROC de cada emoción contra el resto. El **AUC** (Área Bajo la Curva) varía de 0.5 (clasificador aleatorio) a 1.0 (perfecto). También calculamos el **macro-AUC** promedio.

In [ ]:
def compute_roc_data(y_true, y_prob):
    """Calcula curvas ROC y AUC para cada clase (OvR)."""
    y_bin = label_binarize(y_true, classes=list(range(N_CLS)))
    roc = {}
    for i, cls in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc[cls] = {'fpr': fpr, 'tpr': tpr, 'auc': auc(fpr, tpr)}
    macro_auc = np.mean([v['auc'] for v in roc.values()])
    return roc, macro_auc


# Calcular ROC para cada modelo
for name, d in models_data.items():
    d['roc'], d['macro_auc'] = compute_roc_data(d['y_true'], d['y_prob'])

print('ROC calculadas para todos los modelos.')

In [ ]:
# ROC por clase — comparando todos los modelos
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for col, cls in enumerate(CLASSES):
    ax = axes[col]
    ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')

    for name, d in models_data.items():
        roc = d['roc'][cls]
        ax.plot(roc['fpr'], roc['tpr'],
                color=d['color'], lw=1.5,
                label=f"{name.split('(')[0].strip()} (AUC={roc['auc']:.3f})")

    ax.set_title(f'{cls.upper()}\n', fontweight='bold',
                 color=EMOTION_COLORS[cls])
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=6, loc='lower right')
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')

axes[-1].axis('off')
plt.suptitle('Curvas ROC por Emoción — Comparación entre Modelos\n'
             '(One-vs-Rest; AUC más alto = mejor discriminación)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_12_roc_curves_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Macro-AUC comparativo (un gráfico por modelo)
fig, ax = plt.subplots(figsize=(12, 6))

for name, d in models_data.items():
    # Micro-average ROC
    y_bin  = label_binarize(d['y_true'], classes=list(range(N_CLS)))
    fpr_all = np.linspace(0, 1, 200)
    tpr_interp = np.zeros(200)
    for i in range(N_CLS):
        fpr, tpr, _ = roc_curve(y_bin[:, i], d['y_prob'][:, i])
        tpr_interp += np.interp(fpr_all, fpr, tpr)
    tpr_interp /= N_CLS
    macro_auc = auc(fpr_all, tpr_interp)
    ax.plot(fpr_all, tpr_interp, color=d['color'], lw=2,
            label=f"{name} (macro-AUC={d['macro_auc']:.3f})")

ax.plot([0,1],[0,1],'k--',lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Macro-AUC — Curvas ROC promediadas sobre todas las clases', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_13_macro_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 — Matrices de Confusión

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax, normalize=True, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt = '.2f'
    else:
        cm_plot = cm
        fmt = 'd'
    sns.heatmap(
        cm_plot, annot=True, fmt=fmt, cmap=cmap,
        xticklabels=CLASSES, yticklabels=CLASSES,
        ax=ax, cbar=True, linewidths=0.5, linecolor='white',
        vmin=0, vmax=(1 if normalize else None)
    )
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)


# Matrices para los 2 modelos principales (CNN y ResNet18)
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

main_models = [
    ('CNN (MiniEmotionNet)', 'Blues'),
    ('ResNet18 (Transfer Learning)', 'Oranges'),
]
for col, (name, cmap) in enumerate(main_models):
    d = models_data[name]
    plot_confusion_matrix(d['y_true'], d['y_pred'],
                          f'{name}\nNormalizada (recall por fila)',
                          axes[0, col], normalize=True, cmap=cmap)
    plot_confusion_matrix(d['y_true'], d['y_pred'],
                          f'{name}\nAbsoluta (conteos)',
                          axes[1, col], normalize=False, cmap=cmap)

plt.suptitle('Matrices de Confusión — Deep Learning\n'
             'Diagonal principal = predicciones correctas | Off-diagonal = confusiones',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_14_confusion_matrices_dl.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrices para XGBoost
xgb_models = ['XGBoost + CNN features', 'XGBoost + ResNet features', 'XGBoost + RFECV']
cmaps_xgb  = ['Greens', 'Purples', 'Reds']

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, name, cmap in zip(axes, xgb_models, cmaps_xgb):
    d = models_data[name]
    plot_confusion_matrix(d['y_true'], d['y_pred'],
                          name + '\n(normalizada)', ax, normalize=True, cmap=cmap)

plt.suptitle('Matrices de Confusión — XGBoost (features extraídas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_15_confusion_matrices_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análisis de confusiones: ¿qué pares de emociones se confunden más?
d = models_data['CNN (MiniEmotionNet)']
cm = confusion_matrix(d['y_true'], d['y_pred'])

# Off-diagonal: errores entre pares
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

# Top 10 confusiones
pairs = []
for i in range(N_CLS):
    for j in range(N_CLS):
        if i != j and cm_off[i,j] > 0:
            pairs.append({'Real': CLASSES[i], 'Predicho': CLASSES[j], 'Errores': cm_off[i,j]})
df_confusions = pd.DataFrame(pairs).sort_values('Errores', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    [f"{r['Real']} → {r['Predicho']}" for _, r in df_confusions.iterrows()],
    df_confusions['Errores'].values,
    color=[EMOTION_COLORS[r['Real']] for _, r in df_confusions.iterrows()],
    alpha=0.85, edgecolor='white'
)
ax.set_xlabel('Número de errores')
ax.set_title('Top 15 confusiones — MiniEmotionNet (test set)\n'
             'Real → Predicho', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_16_top_confusions.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_confusions.to_string(index=False))

## 4.3 — Radar chart de F1-score por emoción

In [ ]:
def f1_per_class(y_true, y_pred):
    return f1_score(y_true, y_pred, average=None, labels=list(range(N_CLS)))


angles = np.linspace(0, 2*np.pi, N_CLS, endpoint=False).tolist()
angles += angles[:1]  # cerrar el polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})

for name, d in models_data.items():
    f1 = f1_per_class(d['y_true'], d['y_pred']).tolist()
    f1 += f1[:1]
    ax.plot(angles, f1, color=d['color'], lw=2, label=name)
    ax.fill(angles, f1, color=d['color'], alpha=0.06)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.upper() for c in CLASSES], fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax.set_title('Radar Chart — F1-Score por Emoción\n(más alejado del centro = mejor)', 
             fontsize=12, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_17_radar_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 — Tabla comparativa final

In [ ]:
summary_rows = []
for name, d in models_data.items():
    f1 = f1_per_class(d['y_true'], d['y_pred'])
    summary_rows.append({
        'Modelo': name,
        'Accuracy': round(accuracy_score(d['y_true'], d['y_pred']), 4),
        'Macro F1': round(f1_score(d['y_true'], d['y_pred'], average='macro'), 4),
        'Macro AUC': round(d['macro_auc'], 4),
        'F1 angry':    round(f1[0], 3),
        'F1 disgusted':round(f1[1], 3),
        'F1 fearful':  round(f1[2], 3),
        'F1 happy':    round(f1[3], 3),
        'F1 neutral':  round(f1[4], 3),
        'F1 sad':      round(f1[5], 3),
        'F1 surprised':round(f1[6], 3),
    })

df_summary = pd.DataFrame(summary_rows).sort_values('Macro AUC', ascending=False)
df_summary.to_csv(BASE_DIR / 'final_model_comparison.csv', index=False)

# Visualización de la tabla
fig, ax = plt.subplots(figsize=(18, 3))
ax.axis('off')
table_data = df_summary.values
col_labels = df_summary.columns.tolist()
tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.8)

# Colorear header
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2C3E50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Colorear fila ganadora
best_row = df_summary['Macro AUC'].idxmax()
best_pos = df_summary.index.get_loc(best_row)
for j in range(len(col_labels)):
    tbl[best_pos + 1, j].set_facecolor('#D5F5E3')

plt.title('Tabla Comparativa de Modelos — ordenados por Macro-AUC (↓)', 
          fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_18_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTabla guardada en final_model_comparison.csv')
print(df_summary[['Modelo','Accuracy','Macro F1','Macro AUC']].to_string(index=False))

## 4.5 — Análisis de calibración (Reliability Diagram)

Un modelo bien calibrado emite probabilidades que reflejan la frecuencia real de éxito. Si el modelo dice 80% de confianza, debería acertar ~80% de las veces.

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

main_two = ['CNN (MiniEmotionNet)', 'ResNet18 (Transfer Learning)']
for ax, name in zip(axes, main_two):
    d = models_data[name]
    y_bin = label_binarize(d['y_true'], classes=list(range(N_CLS)))

    ax.plot([0,1],[0,1],'k--',lw=1, label='Perfectamente calibrado')
    for i, cls in enumerate(CLASSES):
        try:
            prob_true, prob_pred = calibration_curve(
                y_bin[:,i], d['y_prob'][:,i], n_bins=10, strategy='uniform'
            )
            ax.plot(prob_pred, prob_true, 'o-',
                    color=EMOTION_COLORS[cls], ms=4, lw=1.5, label=cls)
        except Exception:
            pass

    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Fracción de positivos reales')
    ax.set_title(f'Reliability Diagram\n{name}', fontweight='bold')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

plt.suptitle('Calibración de probabilidades — ¿cuánto confiar en las predicciones?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_19_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## Resumen del Paso 4

**Figuras generadas:**
- `fig_12_roc_curves_per_class.png` — ROC por emoción, todos los modelos
- `fig_13_macro_roc.png` — Macro-AUC comparativo
- `fig_14_confusion_matrices_dl.png` — Matrices de confusión Deep Learning
- `fig_15_confusion_matrices_xgb.png` — Matrices de confusión XGBoost
- `fig_16_top_confusions.png` — Top 15 pares de emociones confundidas
- `fig_17_radar_f1.png` — Radar chart F1 por emoción
- `fig_18_comparison_table.png` — Tabla comparativa completa
- `fig_19_calibration.png` — Reliability diagrams

**Conclusiones a analizar en el informe:**
- ¿Qué modelo obtiene mayor Macro-AUC?
- Las clases `disgusted` y `fearful` suelen tener el F1 más bajo (pocas muestras y alta similaridad visual)
- `happy` suele ser la clase con mayor F1 (mayor cantidad de muestras y expresión más distintiva)
- `fearful` y `surprised` son las que más se confunden entre sí (expresiones faciales similares)